# Ex-ante Probabilistic LCA — structured workflow

Presampling → Monte Carlo → Global Sensitivity Analysis on a parameterised Brightway2 model,
in pure Python (no Activity Browser scenario engine).

This is the **structured** variant. Everything case-specific lives in two layers near the top;
the engine below never needs editing. A minimal, all-inline version is in `MC_workflow_simple.ipynb`.

## Layered design — edit top to bottom, but you rarely touch the bottom

| Layer | Section | How often you edit |
|---|---|---|
| **1. Environment binding** | project, databases, functional unit, method | per project |
| **2. Parameter registry** | one declarative table of parameters + distributions | **most often** |
| **3. Presampling** | sample the registry → `PROB_X_t0.csv` | rarely (it's generic) |
| **4. Load X** | read the sample matrix | never |
| **5. Model binding + validation** | index formula exchanges, **auto-check params ↔ formulas** | never |
| **6. Verification + engine** | 5 scenarios vs a known reference | set once per model |
| **7. Full MC run** | engine | never |
| **8. Visualise / 9. GSA** | reporting | per question |

**The one principle that makes it reproducible:** every parameterised exchange is overwritten
from the parameters in every iteration, so the LCA score is a pure function of the parameter
vector — immune to leftover database state from previous runs.

> **Kernel:** use a **numpy < 2** environment (`ab_new`, `ab`, `premise`). `bw2data 3.6.x` crashes
> on numpy ≥ 2 (`np.NaN` removed). The check in Layer 1 warns you.

## Layer 1 — Environment binding

The only Brightway2-specific identifiers. Change these for a new project.

In [ ]:
import ast
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, uniform, triang, lognorm, beta, bernoulli

import bw2data as bd
import brightway2 as bw
from SALib.analyze import delta

if int(np.__version__.split('.')[0]) >= 2:
    print(f"WARNING: numpy {np.__version__}. bw2data 3.6.x needs numpy < 2 "
          f"(LCA crashes with 'np.NaN removed'). Switch to the ab_new / ab / premise kernel.")
else:
    print(f"numpy {np.__version__} OK")

In [ ]:
PROJECT           = "FINAL"
# every database that contains formula (parameterised) exchanges:
PARAMETERISED_DBS = ["SiTaSol_F2v1a", "IEA_PVPS_2020"]
FU_DB, FU_CODE    = "SiTaSol_F2v1a", "ba8e2884bda6251e448284b8d4e9cc15_copy1"
METHOD            = ('ILCD 2.0 2018 midpoint', 'climate change', 'climate change total')

bd.projects.set_current(PROJECT)
fu     = bw.Database(FU_DB).get(FU_CODE)
method = bw.Method(METHOD)

print("Project:", bd.projects.current)
print("FU     :", fu)
print("Method :", method.name)

## Layer 2 — Parameter registry  ← edit here

**Single source of truth.** Add/remove/reorder a parameter by editing one row of `PARAMETERS`.
Everything downstream (sampling, CSV columns, GSA labels, `var_level`) is derived from it.

Rules:
* `name` **must** match the variable used in the Brightway2 exchange `formula`.
* `dist` is a key of `SAMPLERS`; `args` are that sampler's keyword arguments.
* A string `args` value (e.g. `p="pi_FM"`) means *use the already-sampled array of that parameter*
  — this handles dependent draws like a Bernoulli whose probability is itself uncertain.
* `group`: `"project"`/`"activity"` (only labels the exported CSV) — set `None` for intermediate
  helpers (the `pi_*` success-probabilities) so they are excluded from the model output.
* **List order = draw order.** Keep a dependent entry after the helper it references, so the
  random-number stream matches a sequential sampling.

In [ ]:
def pert(size, min, mode, max, shape=4):
    """PERT distribution (beta-based)."""
    a = 1 + shape * (mode - min) / (max - min)
    b = 1 + shape * (max - mode) / (max - min)
    return beta.rvs(a, b, size=size) * (max - min) + min

def _bernoulli(size, p):
    # scalar p -> draw `size`; array p (dependent draw) -> one per element
    return bernoulli.rvs(p, size=size) if np.ndim(p) == 0 else bernoulli.rvs(p)

SAMPLERS = {
    "pert":       lambda size, **a: pert(size, **a),
    "normal":     lambda size, loc, scale: norm.rvs(loc, scale, size=size),
    "uniform":    lambda size, low, width: uniform.rvs(low, width, size=size),    # support [low, low+width]
    "triangular": lambda size, c, loc, scale: triang.rvs(c, loc, scale, size=size),
    "lognormal":  lambda size, s, scale: lognorm.rvs(s, scale=scale, size=size),  # s = sigma of underlying normal; median = scale
    "beta":       lambda size, a, b: beta.rvs(a, b, size=size),
    "bernoulli":  _bernoulli,
}

_S = np.log(1.22)   # geometric SD used for all lognormal inventory parameters

PARAMETERS = [
    # name              dist          args                                               group       description
    dict(name="Eff_PV",        dist="pert",       args=dict(min=0.25, mode=0.28, max=0.31, shape=4), group="project",  desc="Panel efficiency"),
    dict(name="PR_PV",         dist="pert",       args=dict(min=0.8,  mode=0.85, max=0.9,  shape=4), group="project",  desc="Performance ratio"),
    dict(name="LT",            dist="normal",     args=dict(loc=30, scale=5),                        group="project",  desc="Panel lifetime"),
    dict(name="Irrad",         dist="uniform",    args=dict(low=1500, width=500),                    group="project",  desc="Irradiation"),
    dict(name="RT_movpe",      dist="pert",       args=dict(min=0.5, mode=3.5, max=3.5, shape=4),    group="project",  desc="MOVPE runtime"),
    dict(name="P_movpe_tool",  dist="pert",       args=dict(min=1, mode=509, max=509),               group="project",  desc="MOVPE tool power load"),
    dict(name="Zeol_scrub",    dist="triangular", args=dict(c=0, loc=2.55, scale=7.65-2.55),         group="project",  desc="Scrubber granulate"),
    dict(name="Cu_zeol",       dist="pert",       args=dict(min=0.2, mode=0.3, max=0.7, shape=4),    group="project",  desc="Granulate Cu fraction"),
    dict(name="Cu_rec",        dist="bernoulli",  args=dict(p=0.5),                                  group="project",  desc="Cu recycling switch"),
    dict(name="pi_NPsynthCu",  dist="pert",       args=dict(min=0.5, mode=0.7, max=0.8),             group=None,       desc="P(chem synth Cu)"),
    dict(name="bin_NPsynthCu", dist="bernoulli",  args=dict(p="pi_NPsynthCu"),                       group="project",  desc="Cu synthesis choice"),
    dict(name="pi_NPsynthAg",  dist="pert",       args=dict(min=0.5, mode=0.7, max=0.8),             group=None,       desc="P(chem synth Ag)"),
    dict(name="bin_NPsynthAg", dist="bernoulli",  args=dict(p="pi_NPsynthAg"),                       group="project",  desc="Ag synthesis choice"),
    dict(name="pi_CuSint",     dist="pert",       args=dict(min=0.1, mode=0.2, max=0.3, shape=4),    group=None,       desc="P(laser sinter Cu)"),
    dict(name="bin_CuSint",    dist="bernoulli",  args=dict(p="pi_CuSint"),                          group="project",  desc="Cu sintering choice"),
    dict(name="pi_AgSint",     dist="uniform",    args=dict(low=0, width=1),                         group=None,       desc="P(laser sinter Ag)"),
    dict(name="bin_AgSint",    dist="bernoulli",  args=dict(p="pi_AgSint"),                          group="project",  desc="Ag sintering choice"),
    dict(name="pi_FM",         dist="beta",       args=dict(a=4, b=2),                               group=None,       desc="P(Cu vs Ag nanoink)"),
    dict(name="bin_FM",        dist="bernoulli",  args=dict(p="pi_FM"),                              group="project",  desc="Front-metal choice"),
    dict(name="Elec_Siem",     dist="lognormal",  args=dict(s=_S, scale=110),                        group="activity", desc="Siemens electricity"),
    dict(name="Heat_Siem",     dist="lognormal",  args=dict(s=_S, scale=185),                        group="activity", desc="Siemens heat"),
    dict(name="Elec_CZ",       dist="lognormal",  args=dict(s=_S, scale=85.26),                      group="activity", desc="Czochralski electricity"),
    dict(name="scSi_CZ",       dist="lognormal",  args=dict(s=_S, scale=1.07),                       group="activity", desc="Czochralski silicon"),
    dict(name="Al_panel",      dist="lognormal",  args=dict(s=_S, scale=2.63),                       group="project",  desc="Aluminium in panel"),
    dict(name="Glass_panel",   dist="lognormal",  args=dict(s=_S, scale=10.08),                      group="project",  desc="Glass in panel"),
    dict(name="Elec_panel",    dist="lognormal",  args=dict(s=_S, scale=4.71),                       group="project",  desc="Panel manufacturing electricity"),
]

# Model parameters that enter the LCA = everything except the intermediate pi_* helpers (group=None).
MODEL_PARAMS = [p['name'] for p in PARAMETERS if p['group'] is not None]
print(f"{len(PARAMETERS)} registry entries, {len(MODEL_PARAMS)} model parameters:")
print(MODEL_PARAMS)

## Layer 3 — Presampling (registry → CSV)

Generic: draws every registry entry in order, then writes the model parameters to
`PROB_X_t0.csv` in the AB-compatible layout. Skip if you already have the CSV.

In [ ]:
np.random.seed(42)
replications = 10000

sampled = {}
for spec in PARAMETERS:
    args = {k: (sampled[v] if isinstance(v, str) else v) for k, v in spec['args'].items()}
    sampled[spec['name']] = SAMPLERS[spec['dist']](size=replications, **args)

df_samples = pd.DataFrame({n: sampled[n] for n in MODEL_PARAMS})
var_level  = {p['name']: p['group'] for p in PARAMETERS if p['group'] is not None}

df_T = df_samples.T.copy()
df_T.index.name = 'Name'
df_T.insert(0, 'Group', df_T.index.map(var_level))
df_T.to_csv('PROB_X_t0.csv')
print(f"Saved PROB_X_t0.csv  ({replications} replications, {len(MODEL_PARAMS)} parameters)")

## Layer 4 — Load X

Use `PROB_X_t0.csv` (full) or `examples/PROB_X_t0_short.csv` (10 scenarios, fast check).

In [ ]:
CSV_FILE = 'PROB_X_t0.csv'

with open(CSV_FILE, 'r') as f:
    reader = csv.reader(f); next(reader); rows = list(reader)
param_names = [row[0] for row in rows]
X = np.array([row[2:] for row in rows], dtype=float).T   # (replications, n_params)

print(f"File       : {CSV_FILE}")
print(f"X shape    : {X.shape}")
print(f"Parameters : {param_names}")

## Layer 5 — Model binding + validation

Index every formula exchange across **all** parameterised databases, then automatically
cross-check the registry against the formulas. This gate catches the most common mistake —
forgetting a database (so some parameters silently do nothing and get a spurious zero GSA score).

In [ ]:
formula_exchanges = []
for db_name in PARAMETERISED_DBS:
    for act in bw.Database(db_name):
        for exc in act.exchanges():
            if 'formula' in exc:
                formula_exchanges.append(exc)
print(f"Formula exchanges across {PARAMETERISED_DBS}: {len(formula_exchanges)}")

# --- validation gate: parameters declared (loaded from CSV) vs used in formulas ---
declared = set(param_names)
used = set()
for exc in formula_exchanges:
    used |= {n.id for n in ast.walk(ast.parse(exc['formula'], mode='eval')) if isinstance(n, ast.Name)}

unresolved = used - declared       # formulas need a parameter we don't supply -> would keep stale defaults
unused     = declared - used       # parameter affects nothing -> spurious zero GSA sensitivity

if unresolved:
    raise ValueError(f"Formulas reference parameters missing from the CSV/registry: {sorted(unresolved)}")
if unused:
    print(f"\u26a0 Declared but never used by any scanned exchange (check PARAMETERISED_DBS): {sorted(unused)}")
else:
    print("\u2713 Every declared parameter is used by at least one formula exchange.")

In [ ]:
# Introspection table: see exactly how parameters map into the model structure
rows_ = []
for exc in formula_exchanges:
    vs = sorted({n.id for n in ast.walk(ast.parse(exc['formula'], mode='eval')) if isinstance(n, ast.Name)})
    rows_.append({'to activity': exc.output['name'][:38],
                  'input': exc.input['name'][:32],
                  'formula': exc['formula'],
                  'uses': ', '.join(vs)})
model_map = pd.DataFrame(rows_)
model_map

## Layer 6 — The engine + verification

`run_mc` is generic — it never references case-specific names. Verify against a known
reference before the full run.

Reference (reproducible clean-Python values, identical to the original parameter approach):
```
0.136787  0.188661  0.105696  0.148844  0.156320
```

In [ ]:
def run_mc(X, param_names, formula_exchanges, fu, method, n_iter=None):
    """For each scenario i: evaluate every formula exchange from the parameters and run LCA.
    State-independent: all parameterised exchanges are overwritten each iteration."""
    if n_iter is None:
        n_iter = X.shape[0]
    scores = []
    for i in range(n_iter):
        params_i = dict(zip(param_names, X[i].tolist()))
        for exc in formula_exchanges:
            try:
                exc['amount'] = float(eval(exc['formula'], {"__builtins__": None}, params_i))
                exc.save()
            except Exception:
                pass
        lca = bw.LCA({fu: 1}, method.name)
        lca.lci(); lca.lcia()
        scores.append(lca.score)
    return np.array(scores)


ref = [0.136787, 0.188661, 0.105696, 0.148844, 0.156320]
Y_test = run_mc(X, param_names, formula_exchanges, fu, method, n_iter=5)

print(f"{'Sc':>3}  {'Python':>12}  {'reference':>12}  {'Diff':>12}")
print('-' * 46)
for i, (py, rf) in enumerate(zip(Y_test, ref)):
    print(f"{i:>3}  {py:>12.6f}  {rf:>12.6f}  {py-rf:>+12.6f}")

## Layer 7 — Full MC run (10 000 iterations)

Run only after the verification matches.

In [ ]:
import time
t0 = time.time()
Y = run_mc(X, param_names, formula_exchanges, fu, method)
print(f"Done in {time.time()-t0:.0f}s.  n={len(Y)}  mean={Y.mean():.4f}  std={Y.std():.4f}")
pd.DataFrame({'score': Y}).to_csv('Model_results_python.csv', index=False)
print('Saved Model_results_python.csv')

## Layer 8 — Visualise MC results

In [ ]:
# Y = pd.read_csv('Model_results_python.csv')['score'].values   # load if needed

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(Y, bins=50)
axs[0].set_title('Histogram – GWI electricity production')
axs[0].set_xlabel('kg CO\u2082 eq / kWh')
axs[1].boxplot(Y); axs[1].set_title('Boxplot'); axs[1].set_xticks([])
plt.tight_layout(); plt.show()
print(f"Mean={Y.mean():.4f}  Std={Y.std():.4f}  Min={Y.min():.4f}  Max={Y.max():.4f}")

## Layer 9 — GSA (Borgonovo \u03b4)

In [ ]:
problem = {
    'num_vars': X.shape[1],
    'names': param_names,
    'bounds': list(zip(X.min(axis=0), X.max(axis=0)))
}
Si = delta.analyze(problem, X, Y)
df_gsa = Si.to_df()
print(df_gsa.sort_values('delta', ascending=False).to_string())

In [ ]:
plt.figure(figsize=(14, 3))
sns.heatmap(df_gsa[['delta']].T, fmt='.4f', cmap='coolwarm', annot=True, annot_kws={'size': 7},
            cbar_kws={'label': 'Delta sensitivity measure'})
plt.tight_layout(); plt.savefig('GSA_heatmap.png', dpi=150); plt.show()
df_gsa.to_csv('GSA_results.csv')
print('Saved GSA_results.csv and GSA_heatmap.png')